# 01 - Image preprocessing

Optional preprocessing utility used during dataset preparation.

It:
- converts supported image formats (including `.jfif`) to JPEG,
- fixes EXIF orientation,
- crops or pads images to a square,
- resizes to 512×512,
- saves processed images without overwriting the originals.

In [ ]:
!pip -q install pillow

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps
import shutil

INPUT_DIR = Path("/content/fotki")
OUTPUT_DIR = Path("/content/output_photos_jpg")

MODE = "crop"       # "crop" or "pad"
TARGET_SIZE = 512
JPEG_QUALITY = 92
BACKGROUND_COLOR = (0, 0, 0)

VALID_EXT = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff",".jfif"}

def make_square(im, mode="crop"):
    im = ImageOps.exif_transpose(im)

    if im.mode in ("RGBA", "LA"):
        bg = Image.new("RGB", im.size, BACKGROUND_COLOR)
        bg.paste(im, mask=im.split()[-1])
        im = bg
    elif im.mode != "RGB":
        im = im.convert("RGB")

    w, h = im.size
    if w == h:
        return im

    if mode == "crop":
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        return im.crop((left, top, left + side, top + side))

    if mode == "pad":
        side = max(w, h)
        out = Image.new("RGB", (side, side), BACKGROUND_COLOR)
        out.paste(im, ((side-w)//2, (side-h)//2))
        return out

    raise ValueError("MODE must be 'crop' or 'pad'")

def process_one(src, dst):
    with Image.open(src) as im:
        im = make_square(im, MODE)
        if TARGET_SIZE:
            im = im.resize((TARGET_SIZE, TARGET_SIZE), Image.Resampling.LANCZOS)
        dst.parent.mkdir(parents=True, exist_ok=True)
        im.save(dst, "JPEG", quality=JPEG_QUALITY, optimize=True)

files = [p for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXT]
print("Recognized images:", len(files))

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

errors = []
for src in files:
    dst = (OUTPUT_DIR / src.relative_to(INPUT_DIR)).with_suffix(".jpg")
    try:
        process_one(src, dst)
    except Exception as exc:
        errors.append((src, exc))

print("Processed:", len(files) - len(errors))
print("Errors:", len(errors))
for item in errors[:10]:
    print(item)